# Exploratory data analysis — synthetic demo data

A quick tour of the four input files before running the analysis scripts. All data here is synthetic (see `scripts/generate_demo_data.py`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data/demo")
cal = pd.read_csv(DATA / "calibration.csv", parse_dates=["timestamp"])
move = pd.read_csv(DATA / "movement.csv", parse_dates=["timestamp"])
imu = pd.read_csv(DATA / "har_imu.csv", parse_dates=["timestamp"])
stream = pd.read_csv(DATA / "sampling_stream.csv", parse_dates=["timestamp"])
for name, df in [("calibration", cal), ("movement", move), ("har_imu", imu), ("sampling_stream", stream)]:
    print(f"{name:16s} {df.shape}")

## 1. Data quality
Missing values and sampling regularity (time between consecutive readings).

In [ ]:
print(cal.isna().sum().sum(), "missing values in calibration.csv")
gaps = cal.sort_values("timestamp").groupby(["device_id", "session_id"])["timestamp"].diff().dt.total_seconds()
gaps.describe()

## 2. Raw sensor vs reference

In [ ]:
d = cal[cal.device_id == "Device-A"]
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(d.pm25_raw, d.pm25_ref, s=5, alpha=.3)
ax.plot([0, d.pm25_raw.max()], [0, d.pm25_raw.max()], "k--", lw=.8)
ax.set(xlabel="Sensor raw PM2.5", ylabel="Reference PM2.5", title="Device-A")
plt.show()

## 3. Static vs dynamic summary

In [ ]:
(move.groupby(["site", "run_id", "device_id", "phase"])["pm25"]
     .agg(["count", "mean", "std"]).round(2).head(10))

## 4. IMU signal per activity

In [ ]:
import numpy as np
fig, axes = plt.subplots(4, 1, figsize=(8, 7), sharex=True)
for ax, act in zip(axes, ["sitting", "walking", "running", "cycling"]):
    s = imu[(imu.subject_id == "Subject-1") & (imu.activity == act)].head(125)
    ax.plot(np.arange(len(s)) / 25, np.linalg.norm(s[["ax", "ay", "az"]], axis=1))
    ax.set_ylabel(act)
axes[-1].set_xlabel("seconds")
plt.tight_layout(); plt.show()

## Next
Run `python -m scripts.run_all` from the repository root to produce all tables and figures in `outputs/demo/`.